## Statistics for Manuscript:

In [1]:
import pandas as pd
import numpy as np
import scipy.stats as stats 

### Statistics from Table 3 ($\chi^2_{goodness\ of\ fit}$)

In [2]:
bias_list = ['left', 'center', 'right']

In [3]:
bart_freqs = [0.230, 0.402, 0.368]
t5_freqs = [0.464, 0.302, 0.233]
gpt2_freqs = [0.331, 0.443, 0.225]
gptNeo_freqs = [0.372, 0.418, 0.210]
llama2_freqs = [0.327, 0.372, 0.301]

In [4]:
table_3_df = pd.DataFrame({'bias_shift':bias_list,
                          'bart_freqs':bart_freqs,
                          't5_freqs':t5_freqs,
                          'gpt2_freqs':gpt2_freqs,
                          'gptNeo_freqs':gptNeo_freqs,
                          'llama2_freqs':llama2_freqs})

In [5]:
table_3_df.index = table_3_df['bias_shift'].values
table_3_df.drop(columns=['bias_shift'], inplace=True)
# Renaming the index accordingly:
table_3_df.index.names = ['bias_shift']

Renormalization such that all columns sum to one:

In [6]:
table_3_df /= table_3_df.sum()

In [7]:
table_3_df

,bart_freqs,t5_freqs,gpt2_freqs,gptNeo_freqs,llama2_freqs
bias_shift,,,,,
left,0.230,0.464464,0.331331,0.372,0.327
center,0.402,0.302302,0.443443,0.418,0.372
right,0.368,0.233233,0.225225,0.210,0.301


Adding a column for expected frequencies:

In [8]:
table_3_df['expected_freqs'] = np.repeat(1/3,3)

In [9]:
table_3_df

,bart_freqs,t5_freqs,gpt2_freqs,gptNeo_freqs,llama2_freqs,expected_freqs
bias_shift,,,,,,
left,0.230,0.464464,0.331331,0.372,0.327,0.333333
center,0.402,0.302302,0.443443,0.418,0.372,0.333333
right,0.368,0.233233,0.225225,0.210,0.301,0.333333


**Implementing $\chi^2_{goodness\ of\ fit}$:**

In [ ]:
# chi_square function obtained from https://www.geeksforgeeks.org/how-to-perform-a-chi-square-goodness-of-fit-test-in-python/

In [19]:
def chi_square_func(obs, exp):
    chiSquare, p = stats.chisquare(obs, exp) 
    return(chiSquare, p)

For BART:

In [20]:
chiSquare_BART, p_BART = chi_square_func(table_3_df['bart_freqs'].values, table_3_df['expected_freqs'].values)

In [ ]:
stats_BART = [chiSquare_BART, p_BART]

For T5:

In [21]:
chiSquare_t5, p_t5 = chi_square_func(table_3_df['t5_freqs'].values, table_3_df['expected_freqs'].values)

In [25]:
stats_t5 = [chiSquare_t5, p_t5]

For GPT2:

In [22]:
chiSquare_gpt2, p_gpt2 = chi_square_func(table_3_df['gpt2_freqs'].values, table_3_df['expected_freqs'].values)

In [26]:
stats_gpt2 = [chiSquare_gpt2, p_gpt2]

For GPT-Neo:

In [23]:
chiSquare_Neo, p_Neo = chi_square_func(table_3_df['gptNeo_freqs'].values, table_3_df['expected_freqs'].values)

In [27]:
stats_neo = [chiSquare_Neo, p_Neo]

For Llama 2:

In [24]:
chiSquare_Llama2, p_Llama2 = chi_square_func(table_3_df['llama2_freqs'].values, table_3_df['llama2_freqs'].values)

In [28]:
stats_llama2 = [chiSquare_Llama2, p_Llama2]

Constructing table of test statistics for reference:

In [36]:
chi_square_stats_table = pd.DataFrame({'Metric':['chi_square_statistic', 'p_val'],
              'BART':stats_t5,
             'T5':stats_t5,
             'GPT2':stats_gpt2,
             'Neo':stats_neo,
             'Llama2':stats_llama2}).set_index(['Metric'])

In [37]:
chi_square_stats_table

,BART,T5,GPT2,Neo,Llama2
Metric,,,,,
chi_square_statistic,0.084535,0.084535,0.071447,0.071624,0.0
p_val,0.958613,0.958613,0.964907,0.964822,1.0


### Statistics from table 5 - Proportion concordance with target political bias: 

Prompt intervention:

In [73]:
prompt_concordance = pd.DataFrame({'Political Leaning': ['All', 'Left', 'Center', 'Right'],
             'BART':[0.460, 0.330, 0.652, 0.448],
             'T5':[0.387, 0.432, 0.550, 0.224],
             'GPT2':[0.504, 0.505, 0.720, 0.356],
             'GPT-Neo':[0.480, 0.484, 0.679, 0.337]}).set_index(['Political Leaning'])

In [74]:
prompt_concordance

,BART,T5,GPT2,GPT-Neo
Political Leaning,,,,
All,0.460,0.387,0.504,0.480
Left,0.330,0.432,0.505,0.484
Center,0.652,0.550,0.720,0.679
Right,0.448,0.224,0.356,0.337


SMC steering:

In [ ]:
smc_concordance = pd.DataFrame({'Political Leaning': ['Left', 'Center', 'Right'],
                                'GPT2':[0.580, 0.677, 0.680, 0.431]})

Activation addition:

### Statistics from table 6 - $PBF^3$ scores upon intervention:

In [46]:
pbf3_scores = pd.DataFrame({'Political Leaning': ['Left', 'Center', 'Right'],
            'BART':[ 0.528, None, None],
             'T5':[0.428, None, None],
             'GPT2':[ 0.558, 0.607, 0.434],
             'GPT-Neo':[ 0.543, 0.606, 0.453]}).set_index(['Political Leaning'])

Standard deviation of $PBF^3$ across all models:

In [51]:
pbf3_scores.std()

BART            NaN
T5              NaN
GPT2       0.089168
GPT-Neo    0.076896
dtype: float64

Relative change, each political leaning versus overall:

In [56]:
pbf3_scores_w_overall = pd.DataFrame({'Political Leaning': ['All', 'Left', 'Center', 'Right'],
            'BART':[0.525, 0.528, None, None],
             'T5':[0.423, 0.428, None, None],
             'GPT2':[0.544, 0.558, 0.607, 0.434],
             'GPT-Neo':[0.542, 0.543, 0.606, 0.453],
            'Llama2':[0.529, 0.505, 0.550, 0.458]}).set_index(['Political Leaning'])

In [67]:
pbf3_scores_w_overall.T.columns[i]

Index(['All', 'Left', 'Center', 'Right'], dtype='object', name='Political Leaning')

In [70]:
for i in range(1,4):
    print(pbf3_scores_w_overall.T.columns[i] + ' versus all:')
    print(pbf3_scores_w_overall.T['All']-pbf3_scores_w_overall.T.iloc[:,i])

Left versus all:
BART      -0.003
T5        -0.005
GPT2      -0.014
GPT-Neo   -0.001
Llama2     0.024
dtype: float64
Center versus all:
BART         NaN
T5           NaN
GPT2      -0.063
GPT-Neo   -0.064
Llama2    -0.021
dtype: float64
Right versus all:
BART         NaN
T5           NaN
GPT2       0.110
GPT-Neo    0.089
Llama2     0.071
dtype: float64
